In [13]:
# ============================================================
# CELL 1 — Environment, imports, seeding, device, config
# ============================================================

import os, re, ast, math, json, time, random, pickle
from contextlib import nullcontext

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm


# -----------------------
# Reproducibility
# -----------------------
def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
seed_all(SEED)

# -----------------------
# Device
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

if device.type == "cpu":
    try:
        torch.set_num_threads(max(1, os.cpu_count() // 2))
    except Exception:
        pass

# -----------------------
# Mixed precision only on CUDA
# -----------------------
AMP_ENABLED = (device.type == "cuda")
try:
    from torch.amp import autocast, GradScaler
    scaler = GradScaler(enabled=AMP_ENABLED)
    def amp_ctx():
        return autocast(device_type="cuda", dtype=torch.float16, enabled=AMP_ENABLED)
except Exception:
    scaler = None
    def amp_ctx():
        return nullcontext()

# -----------------------
# Data / model / training config
# -----------------------
DATA_PATH = "interactions_ordered.csv"
EMB_PATH  = "emb_movies.pkl"

MAX_LEN = 50
LIMIT_ROWS = 100000   # None => full file

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.7, 0.1, 0.2

BATCH_SIZE = 64 if device.type == "cpu" else 256
NUM_WORKERS = 0
PIN_MEMORY = (device.type == "cuda")

D_MODEL = 768
FINE_TUNE_ITEM_EMB = False

EPOCHS = 13
LR = 2e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

# CPU-friendly sampled training
USE_SAMPLED_SOFTMAX = (device.type == "cpu")
NUM_NEG_TRAIN = 256

# Auxiliary encoder supervision (paper-inspired positional objective)
LAMBDA_POS = 0.20

# Evaluation protocol kept same
EVAL_NEGS = [199, 499, 999]
EVAL_KS = (5, 10)

CKPT_DIR = "pertide_movierec_checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
LAST_CKPT = os.path.join(CKPT_DIR, "pertide_movierec_last.pt")
BEST_CKPT = os.path.join(CKPT_DIR, "pertide_movierec_best.pt")

device: cuda


In [2]:
# ============================================================
# CELL 2 — Load embedding dict + load CSV + robust history parsing
# ============================================================

assert os.path.exists(EMB_PATH), f"Missing embedding file: {EMB_PATH}"
with open(EMB_PATH, "rb") as f:
    emb_dict = pickle.load(f)

assert isinstance(emb_dict, dict), f"Expected dict in {EMB_PATH}, got {type(emb_dict)}"
sample_k = next(iter(emb_dict.keys()))
sample_v = np.asarray(emb_dict[sample_k], dtype=np.float32)

assert sample_v.ndim == 1, f"Expected 1D embedding vectors, got {sample_v.shape}"
D_EXT = int(sample_v.shape[0])

print("Embedding dict size:", len(emb_dict))
print("Sample key:", sample_k, "| dim:", D_EXT, "| dtype:", sample_v.dtype)

# ------------------------------------------------------------
# ID normalization helpers
# Dataset uses m_123 ; embedding dict uses m-123
# We standardize everything internally as m_123.
# ------------------------------------------------------------
def normalize_mid(mid):
    s = str(mid).strip()
    if s.startswith("m_"):
        return s
    if s.startswith("m-"):
        return "m_" + s.split("-", 1)[1]
    if s.startswith("m") and len(s) > 1 and s[1].isdigit():
        return "m_" + s[1:]
    return s

def mid_to_emb_key(mid):
    s = normalize_mid(mid)
    if s in ("<PAD>", "<UNK>"):
        return None
    if s.startswith("m_"):
        return "m-" + s.split("_", 1)[1]
    return s

HIST_ITEM_RE = re.compile(r"(m[_-]\d+)")
HIST_SENT_RE = re.compile(r"\b(pos|neu|neg)\b", re.IGNORECASE)

def parse_history_cell(x):
    """
    Returns list of tuples [(mid, sentiment), ...] in chronological order.

    Handles:
    - python literal strings like "[('m_1','pos'), ('m_2','neg')]"
    - already-materialized Python lists
    - messy strings via regex fallback
    """
    if isinstance(x, list):
        out = []
        for it in x:
            if isinstance(it, (list, tuple)) and len(it) >= 2:
                out.append((normalize_mid(it[0]), str(it[1]).lower()))
        return out

    s = str(x)

    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, list):
            out = []
            for it in obj:
                if isinstance(it, (list, tuple)) and len(it) >= 2:
                    out.append((normalize_mid(it[0]), str(it[1]).lower()))
            if len(out) > 0:
                return out
    except Exception:
        pass

    mids = [normalize_mid(m) for m in HIST_ITEM_RE.findall(s)]
    sents = [z.lower() for z in HIST_SENT_RE.findall(s)]
    if len(sents) < len(mids):
        sents = sents + ["neu"] * (len(mids) - len(sents))

    return [(mids[i], sents[i]) for i in range(min(len(mids), len(sents)))]

assert os.path.exists(DATA_PATH), f"Missing data file: {DATA_PATH}"
df = pd.read_csv(DATA_PATH)

required_cols = {"userid", "history", "target_mid"}
assert required_cols.issubset(df.columns), f"Missing required cols. Found: {df.columns.tolist()}"

if LIMIT_ROWS is not None:
    df = df.iloc[:LIMIT_ROWS].copy()

df["history_parsed"] = df["history"].apply(parse_history_cell)
df["target_mid"] = df["target_mid"].apply(normalize_mid)

# Drop pathological rows with empty parsed history or missing target
df = df[df["history_parsed"].apply(len) > 0].copy()
df = df[df["target_mid"].notna()].copy().reset_index(drop=True)

print("rows after parsing/filtering:", len(df))
print(df.head(3)[["userid", "history_parsed", "target_mid"]])

Embedding dict size: 62423
Sample key: m-1 | dim: 768 | dtype: float32
rows after parsing/filtering: 100000
   userid                                     history_parsed target_mid
0     1.0  [(m_1237, pos), (m_7327, neu), (m_8154, pos), ...     m_7361
1     2.0  [(m_1196, pos), (m_4993, pos), (m_2918, pos), ...     m_2150
2     3.0  [(m_62999, neu), (m_68793, neu), (m_130634, ne...     m_1732


In [3]:
# ============================================================
# CELL 3 — Build vocab + external embedding table W_ext
# ============================================================

PAD_M, UNK_M = 0, 1
PAD_A = 0

SPECIAL_PAD = "<PAD>"
SPECIAL_UNK = "<UNK>"

# Action vocabulary kept exactly aligned with your current dataset structure
action2id = {
    "PAD": 0,
    "pos": 1,
    "neu": 2,
    "neg": 3,
}
id2action = {v: k for k, v in action2id.items()}

# ------------------------------------------------------------
# Collect all item ids that actually appear in the dataset
# ------------------------------------------------------------
all_items = set(df["target_mid"].tolist())
for hist in df["history_parsed"].tolist():
    for mid, _ in hist:
        all_items.add(normalize_mid(mid))
all_items = sorted(list(all_items))

# IMPORTANT FIX:
# Use distinct PAD / UNK tokens.
item2id = {
    SPECIAL_PAD: PAD_M,
    SPECIAL_UNK: UNK_M,
}
for mid in all_items:
    if mid not in item2id:
        item2id[mid] = len(item2id)

id2item = {v: k for k, v in item2id.items()}

num_items = len(item2id)
num_actions = len(action2id)

print("num_items:", num_items)
print("num_actions:", num_actions)

def to_vec(x):
    v = np.asarray(x, dtype=np.float32)
    if v.ndim != 1 or v.shape[0] != D_EXT:
        raise ValueError(f"Bad embedding shape: {v.shape}, expected ({D_EXT},)")
    return v

def build_W_ext(item2id, emb_dict, seed=123):
    """
    Build aligned external embedding matrix W_ext such that:
        W_ext[item2id[m_123]] == emb_dict['m-123']
    """
    rng = np.random.default_rng(seed)
    W = np.zeros((len(item2id), D_EXT), dtype=np.float32)
    missing = 0

    for mid, idx in item2id.items():
        if idx == PAD_M:
            W[idx] = 0.0
            continue
        if idx == UNK_M:
            continue

        emb_key = mid_to_emb_key(mid)
        if emb_key is not None and emb_key in emb_dict:
            W[idx] = to_vec(emb_dict[emb_key])
        else:
            missing += 1
            W[idx] = rng.normal(0.0, 0.02, size=(D_EXT,)).astype(np.float32)

    valid_rows = [i for i in range(len(item2id)) if i not in (PAD_M, UNK_M)]
    if len(valid_rows) > 0:
        W[UNK_M] = W[valid_rows].mean(axis=0)
    else:
        W[UNK_M] = rng.normal(0.0, 0.02, size=(D_EXT,)).astype(np.float32)

    return W, missing

W_ext, missing = build_W_ext(item2id, emb_dict)

print("W_ext shape:", W_ext.shape)
print("missing external vectors:", missing, f"({missing / max(1, num_items - 2):.2%})")

num_items: 28162
num_actions: 4
W_ext shape: (28162, 768)
missing external vectors: 0 (0.00%)


In [4]:
# ============================================================
# CELL 4 — Encode sequences + split train/val/test
# ============================================================

def encode_row(history_parsed, target_mid, max_len=50):
    """
    Convert one raw row into integer-encoded item/action history + target item.

    We keep only the most recent max_len interactions.
    """
    hist = history_parsed[-max_len:] if len(history_parsed) > max_len else history_parsed

    seq_items, seq_acts = [], []
    for mid, sent in hist:
        seq_items.append(item2id.get(normalize_mid(mid), UNK_M))
        seq_acts.append(action2id.get(str(sent).lower(), action2id["neu"]))

    y = item2id.get(normalize_mid(target_mid), UNK_M)
    return seq_items, seq_acts, y

encoded = df.apply(lambda r: encode_row(r["history_parsed"], r["target_mid"], MAX_LEN), axis=1)

df["seq_items"] = [x[0] for x in encoded]
df["seq_acts"]  = [x[1] for x in encoded]
df["y"]         = [x[2] for x in encoded]
df["length"]    = df["seq_items"].apply(len)

# Remove any zero-length rows just in case
df = df[df["length"] > 0].copy().reset_index(drop=True)

print(df["length"].describe())

# ------------------------------------------------------------
# Random split, reproducible
# ------------------------------------------------------------
rng = np.random.RandomState(SEED)
idx = np.arange(len(df))
rng.shuffle(idx)

n_train = int(TRAIN_FRAC * len(df))
n_val   = int(VAL_FRAC * len(df))

train_idx = idx[:n_train]
val_idx   = idx[n_train:n_train + n_val]
test_idx  = idx[n_train + n_val:]

df_train = df.iloc[train_idx].reset_index(drop=True)
df_val   = df.iloc[val_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

print("split sizes:", len(df_train), len(df_val), len(df_test))

count    100000.000000
mean         42.703580
std          10.801793
min          19.000000
25%          34.000000
50%          50.000000
75%          50.000000
max          50.000000
Name: length, dtype: float64
split sizes: 70000 10000 20000


In [5]:
# ============================================================
# CELL 5 — Dataset, collate, and time-mask helpers
# ============================================================

class InteractionsDS(Dataset):
    def __init__(self, frame):
        self.df = frame

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]

        uid = int(float(r["userid"])) if "userid" in r else 0
        items = torch.tensor(r["seq_items"], dtype=torch.long)
        acts  = torch.tensor(r["seq_acts"], dtype=torch.long)
        y     = torch.tensor(r["y"], dtype=torch.long)
        L     = torch.tensor(len(r["seq_items"]), dtype=torch.long)

        return uid, items, acts, L, y

def collate_batch(batch):
    """
    Pads variable-length histories to the batch max length.
    """
    uids, items_list, acts_list, lengths_list, ys = zip(*batch)

    B = len(batch)
    T_max = max(int(L.item()) for L in lengths_list)

    items_pad = torch.full((B, T_max), PAD_M, dtype=torch.long)
    acts_pad  = torch.full((B, T_max), PAD_A, dtype=torch.long)
    lengths   = torch.zeros(B, dtype=torch.long)
    y         = torch.stack(ys, dim=0)
    uid       = torch.tensor(uids, dtype=torch.long)

    for i in range(B):
        L = int(lengths_list[i].item())
        items_pad[i, :L] = items_list[i]
        acts_pad[i, :L]  = acts_list[i]
        lengths[i] = L

    return uid, items_pad, acts_pad, lengths, y

def make_time_mask(lengths, T_max):
    t = torch.arange(T_max, device=lengths.device).unsqueeze(0)
    return t < lengths.unsqueeze(1)

train_loader = DataLoader(
    InteractionsDS(df_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

val_loader = DataLoader(
    InteractionsDS(df_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

test_loader = DataLoader(
    InteractionsDS(df_test),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    collate_fn=collate_batch,
)

print("batches:", len(train_loader), len(val_loader), len(test_loader))

batches: 274 40 79


In [6]:
# ============================================================
# CELL 6 — Negative sampling and ranking metrics
# ============================================================

def sample_negatives(y, num_items, num_neg, gen, start_idx=2):
    """
    Sample negatives in [start_idx, num_items-1], avoiding the true target y.
    """
    B = y.shape[0]
    neg = torch.randint(start_idx, num_items, (B, num_neg), generator=gen, device=y.device)

    for _ in range(3):
        bad = (neg == y.unsqueeze(1))
        if not bad.any():
            break
        neg[bad] = torch.randint(start_idx, num_items, (bad.sum().item(),), generator=gen, device=y.device)

    neg = torch.where(neg == y.unsqueeze(1), ((neg + 1 - start_idx) % (num_items - start_idx)) + start_idx, neg)
    return neg

@torch.no_grad()
def sampled_metrics(scores, true_col=0, K=10):
    """
    scores: [B, 1 + num_neg]
    true item is at column `true_col`
    """
    B = scores.size(0)
    true_scores = scores[:, [true_col]]
    ranks = (scores > true_scores).sum(dim=1) + 1

    mrr = (1.0 / ranks.float()).mean().item()
    hit = (ranks <= K).float().mean().item()

    ndcg = torch.zeros(B, device=scores.device)
    ok = ranks <= K
    ndcg[ok] = 1.0 / torch.log2(ranks[ok].float() + 1.0)

    return mrr, hit, ndcg.mean().item()

@torch.no_grad()
def metrics_from_ranks(ranks, Ks=(5, 10)):
    ranks = ranks.float()
    out = {
        "MRR": (1.0 / ranks).mean().item()
    }
    for K in Ks:
        out[f"Hit@{K}"] = (ranks <= K).float().mean().item()
        ndcg = torch.zeros_like(ranks)
        ok = ranks <= K
        ndcg[ok] = 1.0 / torch.log2(ranks[ok] + 1.0)
        out[f"NDCG@{K}"] = ndcg.mean().item()
    return out

In [7]:
# ============================================================
# CELL 7 — PerTIDE-Rec encoder
# ============================================================

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        norm = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.weight

class PerTIDERec(nn.Module):
    """
    Recommendation-only specialization of PerTIDE.

    Core ideas preserved from the paper:
    1) action-biased b-cells: b_t = mu_t ⊙ e_t
    2) three parallel memory traces:
         - long-term
         - short-term
         - episodic
    3) adaptive fusion of the three traces
    4) task-conditioned gating path (single-task specialization)

    Efficiency notes:
    - Uses diagonal spectral maps instead of dense Q diag(Λ) Q^{-1}
    - Uses sampled softmax for CPU-friendly training
    """

    def __init__(
        self,
        num_items,
        num_actions,
        W_ext,
        d_model=768,
        max_len=50,
        fine_tune_item_emb=False,
        dropout=0.10,
        tie_item_emb=True,
    ):
        super().__init__()

        self.num_items = num_items
        self.num_actions = num_actions
        self.d_model = d_model
        self.max_len = max_len
        self.tie_item_emb = tie_item_emb

        d_ext = int(W_ext.shape[1])
        self.d_ext = d_ext

        # ----------------------------------------------------
        # External item embedding table
        # ----------------------------------------------------
        self.item_emb_raw = nn.Embedding(num_items, d_ext, padding_idx=PAD_M)
        with torch.no_grad():
            self.item_emb_raw.weight.copy_(torch.tensor(W_ext, dtype=torch.float32))
        self.item_emb_raw.weight.requires_grad = bool(fine_tune_item_emb)

        # Projection to model space if external dim != model dim
        if d_ext == d_model:
            self.item_proj = nn.Identity()
        else:
            self.item_proj = nn.Linear(d_ext, d_model, bias=False)

        # ----------------------------------------------------
        # Action-biased b-cell gate: mu_t = sigmoid(MLP(action_onehot))
        # ----------------------------------------------------
        self.action_gate = nn.Sequential(
            nn.Linear(num_actions, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

        # Shared input operator B from the paper's trace equations
        self.B_in = nn.Linear(d_model, d_model, bias=False)

        # ----------------------------------------------------
        # Spectral parameters for the 3 traces
        # Long-term   : exp(delta * lambda_L), lambda_L < 0
        # Short-term  : sigmoid(-kappa * lambda_S), lambda_S > 0
        # Episodic    : 1 + eta * lambda_E, lambda_E < 0, eta small
        # ----------------------------------------------------
        self.lambda_L_raw = nn.Parameter(torch.zeros(d_model))
        self.lambda_S_raw = nn.Parameter(torch.zeros(d_model))
        self.lambda_E_raw = nn.Parameter(torch.zeros(d_model))

        self.delta_raw = nn.Parameter(torch.tensor(0.0))
        self.kappa_raw = nn.Parameter(torch.tensor(0.0))
        self.eta_raw   = nn.Parameter(torch.tensor(-2.0))

        # ----------------------------------------------------
        # Adaptive memory mixer: h_t = MLP([mL_t ; mS_t ; mE_t])
        # ----------------------------------------------------
        self.mix = nn.Sequential(
            nn.LayerNorm(3 * d_model),
            nn.Linear(3 * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

        # ----------------------------------------------------
        # Single-task specialization of task-tuning path
        # The paper's multi-task goal distribution collapses in
        # single-task recommendation, so we keep a learned task
        # embedding + gate conditioned on previous state.
        # ----------------------------------------------------
        self.task_emb = nn.Parameter(torch.randn(d_model) * 0.02)
        self.task_gate = nn.Sequential(
            nn.LayerNorm(2 * d_model),
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.state_bias = nn.Parameter(torch.zeros(d_model))
        self.final_norm = RMSNorm(d_model)

        # Paper-inspired auxiliary position objective
        self.pos_head = nn.Linear(d_model, max_len)

        if not tie_item_emb:
            self.out = nn.Linear(d_model, num_items, bias=False)

    # --------------------------------------------------------
    # Embedding utilities
    # --------------------------------------------------------
    def item_modelspace(self, item_ids):
        return self.item_proj(self.item_emb_raw(item_ids))

    def item_table(self):
        return self.item_proj(self.item_emb_raw.weight)

    def item_table_is_static(self):
        emb_trainable = self.item_emb_raw.weight.requires_grad
        proj_trainable = any(p.requires_grad for p in self.item_proj.parameters())
        return not emb_trainable and not proj_trainable

    # --------------------------------------------------------
    # Spectral gains for the three traces
    # --------------------------------------------------------
    def trace_gains(self):
        # long-term continuous-time eigenvalues < 0
        lambda_L = -(0.05 + 0.95 * torch.sigmoid(self.lambda_L_raw))

        # short-term uses positive spectrum in sigmoid(-kappa * lambda_S)
        lambda_S = 0.25 + 4.75 * torch.sigmoid(self.lambda_S_raw)

        # episodic stable Euler-like factor near identity
        lambda_E = -(0.01 + 0.49 * torch.sigmoid(self.lambda_E_raw))

        delta = F.softplus(self.delta_raw)         # > 0
        kappa = F.softplus(self.kappa_raw)         # > 0
        eta   = 0.25 * torch.sigmoid(self.eta_raw) # small positive

        A_L = torch.exp(delta * lambda_L).clamp(1e-4, 0.9999)
        A_S = torch.sigmoid(-kappa * lambda_S).clamp(1e-4, 0.9999)
        A_E = (1.0 + eta * lambda_E).clamp(1e-4, 0.9999)

        return A_L, A_S, A_E

    # --------------------------------------------------------
    # Encode full sequence
    # --------------------------------------------------------
    def encode(self, items, acts, lengths):
        B, T = items.shape
        mask = make_time_mask(lengths, T)

        # e_t from external embeddings
        e_t = self.item_modelspace(items)  # [B,T,D]

        # mu_t from action one-hot
        a_onehot = F.one_hot(acts, num_classes=self.num_actions).float()
        mu_t = torch.sigmoid(self.action_gate(a_onehot))  # [B,T,D]

        # action-biased b-cell
        b_t = mu_t * e_t  # [B,T,D]

        # shared input operator
        u_t = self.B_in(b_t)  # [B,T,D]

        A_L, A_S, A_E = self.trace_gains()

        mL = torch.zeros(B, self.d_model, device=items.device, dtype=u_t.dtype)
        mS = torch.zeros(B, self.d_model, device=items.device, dtype=u_t.dtype)
        mE = torch.zeros(B, self.d_model, device=items.device, dtype=u_t.dtype)

        z_prev = torch.zeros(B, self.d_model, device=items.device, dtype=u_t.dtype)
        task_vec = self.task_emb.unsqueeze(0).expand(B, -1)

        Z = torch.zeros(B, T, self.d_model, device=items.device, dtype=u_t.dtype)

        for t in range(T):
            mt = mask[:, t].unsqueeze(-1)  # [B,1]
            inp = u_t[:, t, :]            # [B,D]

            newL = A_L * mL + inp
            newS = A_S * mS + inp
            newE = A_E * mE + inp

            mL = torch.where(mt, newL, mL)
            mS = torch.where(mt, newS, mS)
            mE = torch.where(mt, newE, mE)

            h_t = self.mix(torch.cat([mL, mS, mE], dim=-1))
            g_t = torch.sigmoid(self.task_gate(torch.cat([z_prev, task_vec], dim=-1)))
            z_t = self.final_norm(g_t * h_t + self.state_bias)

            z_prev = torch.where(mt, z_t, z_prev)
            Z[:, t, :] = torch.where(mt, z_t, torch.zeros_like(z_t))

        return Z, mask

    def last_hidden(self, Z, lengths):
        idx = (lengths - 1).clamp(min=0)
        return Z[torch.arange(Z.size(0), device=Z.device), idx, :]

    def logits_from_hidden(self, h):
        if self.tie_item_emb:
            E = self.item_table()          # [M,D]
            return h @ E.t()               # [B,M]
        return self.out(h)

    def forward(self, items, acts, lengths, return_all=False):
        Z, mask = self.encode(items, acts, lengths)
        hT = self.last_hidden(Z, lengths)
        if return_all:
            return hT, Z, mask
        return hT

In [8]:
# ============================================================
# CELL 8 — Losses and training/evaluation helpers
# ============================================================

def masked_ce_loss(logits, targets, mask, ignore_index=-100):
    """
    logits  : [B,T,C]
    targets : [B,T]
    mask    : [B,T]
    """
    t = targets.clone()
    t = t.masked_fill(~mask, ignore_index)
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        t.reshape(-1),
        ignore_index=ignore_index
    )

def position_aux_loss(model, Z, lengths):
    """
    Paper-inspired auxiliary position supervision:
    each valid timestep predicts its absolute position in the slice.
    """
    B, T, D = Z.shape
    pos_logits = model.pos_head(Z)  # [B,T,max_len]

    pos_tgt = torch.arange(T, device=Z.device).unsqueeze(0).expand(B, T)
    pos_tgt = pos_tgt.clamp(max=model.max_len - 1)

    mask = make_time_mask(lengths, T)
    return masked_ce_loss(pos_logits, pos_tgt, mask)

def sampled_training_loss_from_hidden(h, y, item_table_or_model, num_neg, gen):
    """
    CPU-friendly training loss:
    compute scores only over [true + sampled negatives] instead of full softmax.
    """
    if torch.is_tensor(item_table_or_model):
        E = item_table_or_model
    else:
        E = item_table_or_model.item_table()

    num_items = E.size(0)
    neg = sample_negatives(y, num_items, num_neg, gen, start_idx=2)
    cand = torch.cat([y.view(-1, 1), neg], dim=1)    # [B,1+neg]
    cand_emb = E[cand]                               # [B,1+neg,D]
    scores = (h.unsqueeze(1) * cand_emb).sum(dim=-1)

    labels = torch.zeros(h.size(0), dtype=torch.long, device=h.device)
    return F.cross_entropy(scores, labels)

@torch.no_grad()
def eval_sampled(model, loader, num_neg=199, Ks=(5, 10), seed=777):
    model.eval()

    gen = torch.Generator(device=device.type).manual_seed(seed)
    E = model.item_table()  # full candidate table, once per eval

    ranks_all = []

    for uid, items, acts, lengths, y in tqdm(loader, desc=f"eval sampled {num_neg}", leave=False):
        items = items.to(device)
        acts = acts.to(device)
        lengths = lengths.to(device)
        y = y.to(device)

        hT = model(items, acts, lengths, return_all=False)

        neg = sample_negatives(y, E.size(0), num_neg, gen, start_idx=2)
        cand = torch.cat([y.view(-1, 1), neg], dim=1)
        cand_emb = E[cand]
        scores = (hT.unsqueeze(1) * cand_emb).sum(dim=-1)

        ranks = 1 + (scores[:, 1:] >= scores[:, :1]).sum(dim=1)
        ranks_all.append(ranks)

    ranks_all = torch.cat(ranks_all, dim=0)
    return metrics_from_ranks(ranks_all, Ks=Ks)

def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

In [9]:
# ============================================================
# CELL 9 — Model init, optimizer, and checkpoint utilities
# ============================================================

model = PerTIDERec(
    num_items=num_items,
    num_actions=num_actions,
    W_ext=W_ext,
    d_model=D_MODEL,
    max_len=MAX_LEN,
    fine_tune_item_emb=FINE_TUNE_ITEM_EMB,
    dropout=0.10,
    tie_item_emb=True,
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_params, trainable_params = count_params(model)
print(f"PerTIDE-Rec total params: {total_params:,}")
print(f"PerTIDE-Rec trainable params: {trainable_params:,}")

def atomic_save(obj, path):
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)

def load_checkpoint_if_available(model, optimizer=None, scaler=None, path=LAST_CKPT):
    if not os.path.exists(path):
        return 1, -1.0

    ckpt = torch.load(path, map_location=device)
    model.load_state_dict(ckpt["model"])

    if optimizer is not None and "optimizer" in ckpt and ckpt["optimizer"] is not None:
        optimizer.load_state_dict(ckpt["optimizer"])

    if scaler is not None and AMP_ENABLED and ("scaler" in ckpt) and (ckpt["scaler"] is not None):
        scaler.load_state_dict(ckpt["scaler"])

    start_epoch = int(ckpt.get("epoch", 0)) + 1
    best_val_mrr = float(ckpt.get("best_val_mrr", -1.0))

    print(f"Resumed from {path} | last epoch={start_epoch - 1} | best_val_mrr={best_val_mrr:.4f}")
    return start_epoch, best_val_mrr

PerTIDE-Rec total params: 26,995,253
PerTIDE-Rec trainable params: 5,366,837


In [10]:
# ============================================================
# CELL 10 — Train one epoch
# ============================================================

def train_one_epoch(model, loader, epoch):
    model.train()

    running_loss = 0.0
    running_next = 0.0
    running_pos = 0.0
    n_steps = 0

    # If the item table is fully static, compute it once per epoch.
    static_E = None
    if model.item_table_is_static():
        with torch.no_grad():
            static_E = model.item_table().detach()

    neg_gen = torch.Generator(device=device.type).manual_seed(SEED + 1000 + epoch)

    pbar = tqdm(loader, desc=f"train ep{epoch}", leave=False)
    for uid, items, acts, lengths, y in pbar:
        items = items.to(device)
        acts = acts.to(device)
        lengths = lengths.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        with amp_ctx():
            hT, Z, mask = model(items, acts, lengths, return_all=True)

            if USE_SAMPLED_SOFTMAX:
                loss_next = sampled_training_loss_from_hidden(
                    hT, y,
                    static_E if static_E is not None else model,
                    NUM_NEG_TRAIN,
                    neg_gen
                )
            else:
                logits = model.logits_from_hidden(hT)
                loss_next = F.cross_entropy(logits, y)

            loss_pos = position_aux_loss(model, Z, lengths)
            loss = loss_next + LAMBDA_POS * loss_pos

        if AMP_ENABLED:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()

        running_loss += float(loss.detach().cpu())
        running_next += float(loss_next.detach().cpu())
        running_pos += float(loss_pos.detach().cpu())
        n_steps += 1

        pbar.set_postfix({
            "L": f"{running_loss / n_steps:.4f}",
            "next": f"{running_next / n_steps:.4f}",
            "pos": f"{running_pos / n_steps:.4f}",
        })

    return {
        "loss": running_loss / max(1, n_steps),
        "loss_next": running_next / max(1, n_steps),
        "loss_pos": running_pos / max(1, n_steps),
    }

In [14]:
# ============================================================
# CELL 11 — Main training loop with validation and checkpointing
# ============================================================

start_epoch, best_val_mrr = load_checkpoint_if_available(
    model=model,
    optimizer=optimizer,
    scaler=scaler,
    path=LAST_CKPT
)

for epoch in range(start_epoch, EPOCHS + 1):
    t0 = time.time()

    train_stats = train_one_epoch(model, train_loader, epoch)
    val_metrics = eval_sampled(model, val_loader, num_neg=EVAL_NEGS[0], Ks=EVAL_KS, seed=SEED + epoch)

    elapsed = time.time() - t0

    print(
        f"[epoch {epoch:02d}] "
        f"train_loss={train_stats['loss']:.4f} "
        f"(next={train_stats['loss_next']:.4f}, pos={train_stats['loss_pos']:.4f}) | "
        f"val MRR={val_metrics['MRR']:.4f} "
        f"Hit@5={val_metrics['Hit@5']:.4f} "
        f"NDCG@5={val_metrics['NDCG@5']:.4f} "
        f"Hit@10={val_metrics['Hit@10']:.4f} "
        f"NDCG@10={val_metrics['NDCG@10']:.4f} | "
        f"time={elapsed:.1f}s"
    )

    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict() if (AMP_ENABLED and scaler is not None) else None,
        "best_val_mrr": best_val_mrr,
        "config": {
            "num_items": num_items,
            "num_actions": num_actions,
            "d_model": D_MODEL,
            "max_len": MAX_LEN,
            "fine_tune_item_emb": FINE_TUNE_ITEM_EMB,
        },
        "item2id": item2id,
        "action2id": action2id,
    }

    epoch_path = os.path.join(CKPT_DIR, f"pertide_rec_epoch_{epoch:02d}.pt")
    atomic_save(ckpt, epoch_path)
    atomic_save(ckpt, LAST_CKPT)

    if val_metrics["MRR"] > best_val_mrr:
        best_val_mrr = val_metrics["MRR"]
        ckpt["best_val_mrr"] = best_val_mrr
        atomic_save(ckpt, BEST_CKPT)
        print(f"new best checkpoint saved -> {BEST_CKPT} | best_val_mrr={best_val_mrr:.4f}")

print("training complete.")

Resumed from pertide_movierec_checkpoints/pertide_movierec_last.pt | last epoch=8 | best_val_mrr=0.3988


train ep9:   0%|          | 0/274 [00:00<?, ?it/s]

eval sampled 199:   0%|          | 0/40 [00:00<?, ?it/s]

[epoch 09] train_loss=7.6391 (next=7.5512, pos=0.4394) | val MRR=0.4075 Hit@5=0.5626 NDCG@5=0.4223 Hit@10=0.7001 NDCG@10=0.4670 | time=16.8s
new best checkpoint saved -> pertide_movierec_checkpoints/pertide_movierec_best.pt | best_val_mrr=0.4075


train ep10:   0%|          | 0/274 [00:00<?, ?it/s]

...

eval sampled 199:   0%|          | 0/40 [00:00<?, ?it/s]

[epoch 12] train_loss=7.2859 (next=7.2108, pos=0.3754) | val MRR=0.3951 Hit@5=0.5499 NDCG@5=0.4094 Hit@10=0.6880 NDCG@10=0.4542 | time=15.5s


train ep13:   0%|          | 0/274 [00:00<?, ?it/s]

eval sampled 199:   0%|          | 0/40 [00:00<?, ?it/s]

[epoch 13] train_loss=7.1498 (next=7.0757, pos=0.3707) | val MRR=0.3859 Hit@5=0.5387 NDCG@5=0.3996 Hit@10=0.6756 NDCG@10=0.4440 | time=15.5s
training complete.


In [15]:
# ============================================================
# CELL 12 — Load best checkpoint and evaluate on test with same eval negs
# ============================================================

assert os.path.exists(BEST_CKPT), f"Missing best checkpoint: {BEST_CKPT}"

best_ckpt = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(best_ckpt["model"])
model.eval()

print(f"Loaded best epoch: {best_ckpt.get('epoch', 'NA')}")
print(f"Best val MRR: {best_ckpt.get('best_val_mrr', -1.0):.4f}")

test_results = {}
for nneg in EVAL_NEGS:
    res = eval_sampled(model, test_loader, num_neg=nneg, Ks=EVAL_KS, seed=SEED + 999 + nneg)
    test_results[nneg] = res
    print(f"\nTEST @ {nneg} negatives")
    print(json.dumps(res, indent=2))

print("\nAll test results:")
print(json.dumps(test_results, indent=2))

Loaded best epoch: 9
Best val MRR: 0.4075


eval sampled 199:   0%|          | 0/79 [00:00<?, ?it/s]


TEST @ 199 negatives
{
  "MRR": 0.4055578410625458,
  "Hit@5": 0.5612999796867371,
  "NDCG@5": 0.42071741819381714,
  "Hit@10": 0.6951499581336975,
  "NDCG@10": 0.4641079604625702
}


eval sampled 499:   0%|          | 0/79 [00:00<?, ?it/s]


TEST @ 499 negatives
{
  "MRR": 0.2879110276699066,
  "Hit@5": 0.4022499918937683,
  "NDCG@5": 0.29131028056144714,
  "Hit@10": 0.5287500023841858,
  "NDCG@10": 0.3321961462497711
}


eval sampled 999:   0%|          | 0/79 [00:00<?, ?it/s]


TEST @ 999 negatives
{
  "MRR": 0.2178386002779007,
  "Hit@5": 0.3009999990463257,
  "NDCG@5": 0.21500571072101593,
  "Hit@10": 0.4096499979496002,
  "NDCG@10": 0.2500518560409546
}

All test results:
{
  "199": {
    "MRR": 0.4055578410625458,
    "Hit@5": 0.5612999796867371,
    "NDCG@5": 0.42071741819381714,
    "Hit@10": 0.6951499581336975,
    "NDCG@10": 0.4641079604625702
  },
  "499": {
    "MRR": 0.2879110276699066,
    "Hit@5": 0.4022499918937683,
    "NDCG@5": 0.29131028056144714,
    "Hit@10": 0.5287500023841858,
    "NDCG@10": 0.3321961462497711
  },
  "999": {
    "MRR": 0.2178386002779007,
    "Hit@5": 0.3009999990463257,
    "NDCG@5": 0.21500571072101593,
    "Hit@10": 0.4096499979496002,
    "NDCG@10": 0.2500518560409546
  }
}
